## TIME and TimeType examples

* `TIME` is available in Databricks Runtime 18+ and is currently in Beta.
* It represents a time of day without a date or timezone.
* If any statement fails with a preview-related error, enable the preview for `TIME` first.

In [0]:
%sql
SELECT
  TIME '09:15:00' AS opening_time,
  TIME '14:30:15.123456' AS precise_time,
  current_time() AS current_time_value,
  make_time(23, 59, 59.5) AS end_of_day_sample

In [0]:
parsed = spark.sql("""
SELECT
  CAST('08:45:00' AS TIME) AS cast_time,
  to_time('6:05 PM') AS parsed_12h,
  to_time('18:05:07.25') AS parsed_fractional,
  try_to_time('not-a-time') AS invalid_returns_null
""")

display(parsed)

In [0]:
ops = spark.sql("""
WITH business_hours AS (
  SELECT * FROM VALUES
    ('A', TIME '08:30:00', TIME '17:15:45'),
    ('B', TIME '09:00:00', TIME '18:05:15'),
    ('C', TIME '13:10:30', TIME '16:40:10')
  AS t(shift_id, start_time, end_time)
)
SELECT
  shift_id,
  start_time,
  end_time,
  time_diff(end_time, start_time) AS duration_seconds,
  time_trunc('HOUR', start_time) AS start_hour,
  CASE
    WHEN start_time BETWEEN TIME '09:00:00' AND TIME '17:00:00' THEN 'business_window'
    ELSE 'outside_window'
  END AS start_bucket
FROM business_hours
ORDER BY shift_id
""")

display(ops)

In [0]:
from pyspark.sql import types as T

from_ts = spark.sql("""
WITH samples AS (
  SELECT * FROM VALUES
    (1, TIMESTAMP_NTZ '2026-07-05 08:15:30'),
    (2, TIMESTAMP_NTZ '2026-07-05 21:45:10')
  AS t(id, ts_ntz)
)
SELECT
  id,
  ts_ntz,
  CAST(ts_ntz AS TIME) AS time_only
FROM samples
ORDER BY id
""")

print("Python TimeType available:", hasattr(T, "TimeType"))
if hasattr(T, "TimeType"):
    print("Example Python TimeType object:", T.TimeType())

display(from_ts)